# Knowledge Graphs and Link Prediction

## Learning Objectives

1. **Define** knowledge graphs and triple stores
2. **Explain** the TransE embedding model for link prediction
3. **Implement** TransE training and scoring
4. **Describe** RotatE and other geometric KG embedding models

## Knowledge Graphs

A **knowledge graph** (KG) is a set of triples $(h, r, t)$:
- $h$ = head entity (e.g., "Paris")
- $r$ = relation (e.g., "capitalOf")
- $t$ = tail entity (e.g., "France")

**Tasks:**
- **Link prediction:** given $(h, r, ?)$, find $t$ — entity completion
- **Triple classification:** is $(h, r, t)$ true or false?
- **Relation extraction:** extract new triples from text

**Examples:** Freebase, Wikidata, YAGO, DBpedia, Google Knowledge Graph.

## TransE: Translating Embeddings

**Idea (Bordes et al. 2013):** embed entities $e_h, e_t \in \mathbb{R}^d$ and relations $e_r \in \mathbb{R}^d$ such that:
$$e_h + e_r \approx e_t \text{ for true triple } (h, r, t)$$

**Scoring function:**
$$f(h, r, t) = -\|e_h + e_r - e_t\|$$

Higher score (less negative) = more likely true triple.

**Training loss (pairwise margin):**
$$L = \sum_{(h,r,t) \in T^+} \sum_{(h',r,t') \in T^-} \max(0, \gamma + f(h,r,t') - f(h,r,t))$$

where $T^-$ = corrupted triples (replace $h$ or $t$ with random entity).

In [1]:
import math, random

def norm(v): return math.sqrt(sum(x**2 for x in v))
def normalize(v): n=norm(v); return [x/n for x in v] if n>0 else v
def vec_sub(a, b): return [x-y for x,y in zip(a,b)]
def vec_add(a, b): return [x+y for x,y in zip(a,b)]
def l2(v): return math.sqrt(sum(x**2 for x in v))

def transe_score(eh, er, et): return -l2(vec_sub(vec_add(eh, er), et))

class TransE:
    def __init__(self, entities, relations, d=5, gamma=1.0, eta=0.01, seed=0):
        rng = random.Random(seed)
        self.d = d; self.gamma = gamma; self.eta = eta
        self.ent_emb = {e: normalize([rng.gauss(0,1) for _ in range(d)]) for e in entities}
        self.rel_emb = {r: normalize([rng.gauss(0,1) for _ in range(d)]) for r in relations}

    def score(self, h, r, t):
        return transe_score(self.ent_emb[h], self.rel_emb[r], self.ent_emb[t])

    def train_step(self, pos_triples, entities_list):
        rng = random.Random()
        for h, r, t in pos_triples:
            # Corrupt head or tail
            if rng.random() < 0.5:
                h_neg = rng.choice(entities_list)
                neg = (h_neg, r, t)
            else:
                t_neg = rng.choice(entities_list)
                neg = (h, r, t_neg)

            score_pos = self.score(h, r, t)
            score_neg = self.score(*neg)
            loss = max(0, self.gamma - score_pos + score_neg)
            if loss <= 0: continue

            # Gradient descent (simplified subgradient)
            eta = self.eta
            diff_pos = vec_sub(vec_add(self.ent_emb[h], self.rel_emb[r]), self.ent_emb[t])
            diff_neg = vec_sub(vec_add(self.ent_emb[neg[0]], self.rel_emb[r]), self.ent_emb[neg[2]])
            n_pos = max(l2(diff_pos), 1e-8); n_neg = max(l2(diff_neg), 1e-8)
            # Update entities and relation (sign of gradient)
            for i in range(self.d):
                g_pos = diff_pos[i]/n_pos; g_neg = diff_neg[i]/n_neg
                self.ent_emb[h][i]       -= eta * (-g_pos)
                self.ent_emb[t][i]       -= eta * g_pos
                self.rel_emb[r][i]       -= eta * (-g_pos + g_neg)
                self.ent_emb[neg[0]][i]  += eta * g_neg  # move away
                self.ent_emb[neg[2]][i]  -= eta * (-g_neg)
            # Renormalize entity embeddings
            for e in [h, t, neg[0], neg[2]]:
                self.ent_emb[e] = normalize(self.ent_emb[e])

# Simple KG: countries and capitals
entities = ['Paris','France','Berlin','Germany','London','UK','Madrid','Spain','Europe']
relations = ['capitalOf','locatedIn']
triples = [
    ('Paris','capitalOf','France'),
    ('Berlin','capitalOf','Germany'),
    ('London','capitalOf','UK'),
    ('Madrid','capitalOf','Spain'),
    ('France','locatedIn','Europe'),
    ('Germany','locatedIn','Europe'),
]

model = TransE(entities, relations, d=3, gamma=1.0, eta=0.05)
for epoch in range(200):
    model.train_step(triples, entities)

print("TransE link prediction scores:")
print("True triples:")
for h,r,t in triples[:4]:
    print(f"  ({h}, {r}, {t}): {model.score(h,r,t):.4f}")
print("""
False triples:""")
for h,r,t in [('Paris','capitalOf','Germany'),('Berlin','capitalOf','UK')]:
    print(f"  ({h}, {r}, {t}): {model.score(h,r,t):.4f}")

TransE link prediction scores:
True triples:
  (Paris, capitalOf, France): -3.4098
  (Berlin, capitalOf, Germany): -3.4099
  (London, capitalOf, UK): -3.3537
  (Madrid, capitalOf, Spain): -3.3907

False triples:
  (Paris, capitalOf, Germany): -3.4096
  (Berlin, capitalOf, UK): -3.3427


## Beyond TransE: Geometric Models

| Model | Transformation | Can model |
|-------|---------------|-----------|
| **TransE** | $e_h + e_r \approx e_t$ | 1-to-1 relations |
| **TransR** | Relation-specific projection | 1-to-N, N-to-1 |
| **DistMult** | $e_h \odot e_r \cdot e_t$ | Symmetric relations |
| **ComplEx** | Complex embeddings | Asymmetric + antisymmetric |
| **RotatE** | Rotation in complex plane | All relation patterns |

**RotatE (Sun et al. 2019):** $e_h \circ e_r = e_t$ where $\circ$ = element-wise complex multiplication ($|e_r| = 1$).
Models composition, symmetry, antisymmetry, inversion simultaneously.